# 11 — Amari's Bridge

The identity the whole course has been walking toward: the Sinkhorn plan is the **KL projection** of the Gibbs kernel onto the transportation polytope. Transport geometry *is* information geometry.

**Prerequisites:** `08_kantorovich_duality`, `09_entropic_regularization`, `10_sinkhorn_algorithm`, and `02/02_kl_divergence`, `02/07_fisher_rao_metric`.

**What you'll be able to do**
- State the KL-projection identity, the bridge this notebook is named for: the entropic OT plan is the KL projection of the Gibbs kernel onto $T(a, b)$.
- Check the projection on an example — every marginal-preserving perturbation you try increases the KL — and see why convexity, not the check, is the proof.
- Read how this closes the M2 (information) ↔ M3 (transport) loop, and what of it Module 4 carries to quantum states.

You have a fast algorithm (10). Here is the payoff: a single identity that explains *why* it works, and reveals that the transport geometry you have been building is the information geometry of Module 2 wearing a different hat. This is the bridge the course has been heading toward — and the identity itself carries over to quantum states, as `04/08` shows.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost
from qot_course.transport.sinkhorn import gibbs_kernel, sinkhorn, kl_to_kernel

np.random.seed(42)
viz.use_course_style()

## Where we are

Three bricks brought us here: `08_kantorovich_duality` gave the dual face and its potentials, whose entropic counterparts `03/09` turned into the scalings $u, v$, `09_entropic_regularization` added the entropy bonus and the Gibbs kernel $K = e^{-C/\varepsilon}$, and `10_sinkhorn_algorithm` solved it in five lines. Now we ask *why* that iteration is the right thing to do — and the answer reaches back to the KL divergence of Module 2.

## 1. Sinkhorn is a KL projection

Plug the Gibbs kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ into the KL divergence and expand:

$$ \varepsilon\, \mathrm{KL}(P \,\|\, K) = \sum_{ij} P_{ij}\bigl(\varepsilon \log P_{ij} + C_{ij}\bigr) + \mathrm{const} = \langle C, P\rangle - \varepsilon\, H(P) + \mathrm{const}. $$

The right-hand side is exactly the entropic OT objective. So **minimising the entropic cost over $T(a, b)$ is the same as minimising $\mathrm{KL}(P \,\|\, K)$ over $T(a, b)$** — the Sinkhorn plan is the **KL projection of the Gibbs kernel onto the transportation polytope**. Cuturi (2013) regularised the transport problem with entropy and solved it by Sinkhorn scaling; Benamou, Carlier, Cuturi, Nenna and Peyré (2015) wrote the regularised problem as this KL projection and read the iteration itself as iterated Bregman projections under KL: each half-step is the KL projection onto one marginal constraint, which is the multiplicative row/column rescaling you already know. The course calls the identity *Amari's bridge* because it lands in the information geometry of the KL divergence that Amari developed (Amari, 2016); Amari, Karakida and Oizumi (2018) later studied the information geometry of these entropic plans directly.

If $P^\star_\varepsilon$ truly is that projection, then *any* marginal-preserving move away from it must increase the KL. Let's check.

In [ ]:
eps = 0.4
positions = np.linspace(0.0, 9.0, 16)

def bump(center, sigma=1.2):
    p = np.exp(-0.5 * ((positions - center) / sigma) ** 2)
    return p / p.sum()

src = bump(2.0)
tgt = bump(7.0)
C_demo = squared_euclidean_cost(positions, positions)

K = gibbs_kernel(C_demo, eps)
plan_star = sinkhorn(src, tgt, C_demo, epsilon=eps, n_iter=5000, tol=1e-14)
kl_star = kl_to_kernel(plan_star, K)

# Marginal-preserving 2x2-cycle perturbations: add delta*[[+1,-1],[-1,+1]] on a sub-block.
rng = np.random.default_rng(7)
perturbed_kls = []
while len(perturbed_kls) < 6:
    i1, i2 = rng.choice(plan_star.shape[0], 2, replace=False)
    j1, j2 = rng.choice(plan_star.shape[1], 2, replace=False)
    delta = 0.5 * min(plan_star[i1, j2], plan_star[i2, j1])
    if delta < 1e-12:
        continue
    P = plan_star.copy()
    P[i1, j1] += delta; P[i2, j2] += delta
    P[i1, j2] -= delta; P[i2, j1] -= delta
    if np.all(P > 0):
        perturbed_kls.append(kl_to_kernel(P, K))

print(f"KL(P*_eps || K) at the Sinkhorn fixed point   = {kl_star:.6f}")
print(f"min KL among {len(perturbed_kls)} marginal-preserving moves = {min(perturbed_kls):.6f}")
print(f"every perturbation increases the KL?  {all(k > kl_star for k in perturbed_kls)}")

In [ ]:
increases = np.array([k - kl_star for k in perturbed_kls])
print("KL increase of each move:", ", ".join(f"{x:.1e}" for x in increases))
print(f"smallest {increases.min():.1e}, largest {increases.max():.2f}")

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(range(len(increases)), increases, color=viz.SOURCE_COLOR)
ax.set_yscale("log")   # the increases span many orders of magnitude
ax.set_xticks(range(len(increases)))
ax.set_xticklabels([f"move {i + 1}" for i in range(len(increases))], fontsize=9)
ax.set_ylabel("KL(P || K)  -  KL(P* || K)   (log scale)")
ax.set_title("Every marginal-preserving move rises above the Sinkhorn plan's KL", pad=12)
plt.show()

**Read the output and figure.** Every perturbed coupling we tried has a larger $\mathrm{KL}(P \,\|\, K)$ than the Sinkhorn fixed point. The log axis matters here: the six increases run from about $2 \times 10^{-10}$ to $0.49$, more than nine orders of magnitude, and on a linear axis five of the six bars would be invisible. The Sinkhorn plan itself sits at zero, which a log axis cannot draw.

Six moves are a check on this example, not a proof of minimality. The proof is the expansion above together with convexity. $\mathrm{KL}(\cdot \,\|\, K)$ is strictly convex, so a positive coupling minimises it over $T(a, b)$ exactly when $\log(P_{ij}/K_{ij})$ splits as $x_i + y_j$, one term per row and one per column. The Sinkhorn plan $u_i K_{ij} v_j$ has that form, with $x_i = \log u_i$ and $y_j = \log v_j$. So $P^\star_\varepsilon$ *is* the KL projection of the Gibbs kernel onto $T(a, b)$, and the six moves agree with it.

Why this is more than a reformulation:
- **It generalises.** Replace KL by any Bregman divergence and the same projection logic applies. In Module 4 the *Umegaki relative entropy* replaces the classical KL: the projection identity then holds for every pair of quantum states, while the operator Sinkhorn iteration is sure to reach that projection when the operators commute and can miss it otherwise (`04/08`).
- **It explains the speed.** Alternating Bregman projections onto affine constraints converge linearly under KL — the geometric rate of `03/10`.
- **It unifies the course.** The transport geometry of M3 and the information geometry of M2 are the same geometry, approached from two sides.

## 2. Dictionary update — closing the M2 ↔ M3 loop

The Sinkhorn arc adds the duality and entropic rows, and the last one is the bridge itself. In Module 4 the classical KL projection becomes the Umegaki projection, an identity that holds for every pair of states, while the operator Sinkhorn iteration is sure to reach that projection in the commuting case and can miss it outside (`04/08`).

**New in the dictionary.** The complete table lives in [the course dictionary](../dictionary.ipynb) (`notebooks/dictionary.ipynb`).

| Classical | Quantum |
| --- | --- |
| Kantorovich potentials $\varphi, \psi$ (LP duals) | quantum potentials (SDP duals; not built in this course) |
| Gibbs kernel $K_{ij} = e^{-C_{ij}/\varepsilon}$ (entrywise) | matrix-exponential kernel $K = \exp(-C/\varepsilon)$ |
| Sinkhorn iteration (row/column matrix scaling) | quantum Sinkhorn (operator scaling: $(X\otimes Y)\,K\,(X\otimes Y)$, partial-trace fixed point) |
| entropic plan = KL projection of $K = e^{-C/\varepsilon}$ (computed by Sinkhorn) | entropic quantum plan = Umegaki projection of $K = \exp(-C/\varepsilon)$ (reached by operator Sinkhorn in the commuting case) |

M2 (information geometry) and M3 (transport geometry) meet here, and the identity, though not always the algorithm that computes it, lifts to quantum states.

## Your turn

1. **Do the algebra.** Expand $\varepsilon\,\mathrm{KL}(P \,\|\, K)$ with $K = e^{-C/\varepsilon}$ and confirm it equals $\langle C, P\rangle - \varepsilon H(P) + \mathrm{const}$. What exactly is the constant, and why does it not affect the minimiser over $T(a, b)$?
2. **Projections, half-step by half-step.** Run a few Sinkhorn iterations and check that each half-step exactly fixes one marginal (a row rescale makes $P\mathbf{1} = a$; a column rescale makes $P^\top\mathbf{1} = b$) — alternating Bregman projections in action.
3. **Toward the quantum bridge.** Which Module-2 quantity is the quantum analogue of the KL divergence that will define the entropic quantum plan of Module 4? Find where you met it (hint: `02/10`) and state the projection it will solve.

## What you built

- You stated the KL-projection identity of Cuturi (2013) and Benamou et al. (2015), the bridge this notebook is named for, and proved by expansion that entropic OT minimises $\mathrm{KL}(P\|K)$ over $T(a, b)$.
- You checked the projection on an example — each of six marginal-preserving moves increases the KL — and saw that convexity, not the six moves, makes it a proof.
- You closed the M2 ↔ M3 loop in the dictionary and saw what of the bridge Module 4 carries to quantum states.

This is one of the deepest ideas in the course: the transport geometry of Module 3 and the information geometry of Module 2 are a single geometry seen from two angles. From here, the entropic side of quantum optimal transport is one more change of object — swap probability vectors for density matrices, and KL for Umegaki. The projection identity survives that swap for every pair of states; the Sinkhorn iteration is sure to reach the projection when the operators commute and can miss it otherwise (`04/08`).

## What's next

The last arc of Module 3 lifts transport to a continuous family where everything is closed-form. In `12_bures_wasserstein` we compute $W_2$ between Gaussians without any LP — and that formula is the literal doorway into quantum optimal transport.

## References

- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016). DOI:10.1007/978-4-431-55978-8
- S.-i. Amari, R. Karakida & M. Oizumi, "Information geometry connecting Wasserstein distance and Kullback–Leibler divergence via the entropy-relaxed transportation problem", *Information Geometry* 1, 13–37 (2018). DOI:10.1007/s41884-018-0002-8
- J.-D. Benamou, G. Carlier, M. Cuturi, L. Nenna & G. Peyré, "Iterative Bregman projections for regularized transportation problems", *SIAM J. Sci. Comput.* 37, A1111–A1138 (2015). DOI:10.1137/141000439
- M. Cuturi, "Sinkhorn distances: lightspeed computation of optimal transport", *NeurIPS* (2013).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 4–5. DOI:10.1561/2200000073
- R. Sinkhorn, "A relationship between arbitrary positive matrices and doubly stochastic matrices", *Ann. Math. Statist.* 35, 876–879 (1964).

**Previous:** `notebooks/03_ClassicalOptimalTransport/10_sinkhorn_algorithm.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/12_bures_wasserstein.ipynb`